In [2]:
!pip install cadquery

In [3]:
!pip install trimesh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 12.0 MB/s eta 0:00:00


In [5]:
from best_iou import get_iou_best
from valid_syntax_rate import evaluate_syntax_rate_simple

In [8]:
import torch

In [6]:
## Example usage of the metrics
sample_code = """
height = 60.0
width = 80.0
thickness = 10.0
diameter = 22.0

# make the base
result = (
    cq.Workplane("XY")
    .box(height, width, thickness)
)
"""

sample_code_2 = """
 height = 60.0
 width = 80.0
 thickness = 10.0
 diameter = 22.0
 padding = 12.0

 # make the base
 result = (
     cq.Workplane("XY")
     .box(height, width, thickness)
     .faces(">Z")
     .workplane()
     .hole(diameter)
     .faces(">Z")
     .workplane()
     .rect(height - padding, width - padding, forConstruction=True)
     .vertices()
     .cboreHole(2.4, 4.4, 2.1)
 )
"""
codes = {
    "sample_code": sample_code,
    "sample_code_2": sample_code_2,
}
vsr = evaluate_syntax_rate_simple(codes)
print("Valid Syntax Rate:", vsr)
iou = get_iou_best(sample_code, sample_code_2)
print("IOU:", iou)

Valid Syntax Rate: 1.0
IOU: 0.5834943417057687


In [10]:
d = torch.load("eval.pt")
gt = {f"s{i}": c for i, c in enumerate(d["codes"][:100])}
print("GT VSR:", evaluate_syntax_rate_simple(gt))

GT VSR: 1.0


In [17]:
p = torch.load("preds_a.pt")
pred = {f"s{i}": c for i, c in enumerate(p["preds"])}
print("VSR:", evaluate_syntax_rate_simple(pred))

VSR: 0.51


In [19]:
p = torch.load("preds_b.pt")
pred = {f"s{i}": c for i, c in enumerate(p["preds"])}
print("VSR:", evaluate_syntax_rate_simple(pred))

VSR: 1.0


In [15]:
for i, c in enumerate(p["preds"][:20]):
    try:
        ns = {"cq": cq}; exec(c, ns)
    except Exception as e:
        print(i, type(e).__name__, str(e)[:80])

0 SyntaxError '(' was never closed (<string>, line 13)
3 SyntaxError '(' was never closed (<string>, line 13)
4 SyntaxError '(' was never closed (<string>, line 13)
8 SyntaxError '(' was never closed (<string>, line 13)
13 SyntaxError '(' was never closed (<string>, line 14)
14 SyntaxError '(' was never closed (<string>, line 14)
17 SyntaxError '(' was never closed (<string>, line 14)
18 SyntaxError '(' was never closed (<string>, line 13)


In [13]:
import numpy as np
ious = []
for g, r in zip(p["gt"], p["preds"]):
    try: ious.append(get_iou_best(g, r))
    except Exception: ious.append(0.0)
print("mean:", np.mean(ious), "median:", np.median(ious))

mean: 0.03383589769497823 median: 0.002255072656187921


In [21]:
import numpy as np
ious = []
for g, r in zip(p["gt"], p["preds"]):
    try:
      ious.append(get_iou_best(g, r))
      print(ious)
    except Exception: ious.append(0.0)
print("mean:", np.mean(ious), "median:", np.median(ious))

[np.float64(0.018735904701324236)]
[np.float64(0.018735904701324236), np.float64(0.01658374792703151)]
[np.float64(0.018735904701324236), np.float64(0.01658374792703151), np.float64(0.03612167300380228)]
[np.float64(0.018735904701324236), np.float64(0.01658374792703151), np.float64(0.03612167300380228), np.float64(0.0065165037540728145)]
[np.float64(0.018735904701324236), np.float64(0.01658374792703151), np.float64(0.03612167300380228), np.float64(0.0065165037540728145), np.float64(0.010518114530580444)]
[np.float64(0.018735904701324236), np.float64(0.01658374792703151), np.float64(0.03612167300380228), np.float64(0.0065165037540728145), np.float64(0.010518114530580444), np.float64(0.051264367816091956)]
[np.float64(0.018735904701324236), np.float64(0.01658374792703151), np.float64(0.03612167300380228), np.float64(0.0065165037540728145), np.float64(0.010518114530580444), np.float64(0.051264367816091956), np.float64(0.04307201458523245)]
[np.float64(0.018735904701324236), np.float64(0.0

In [22]:
print(len(set(p["preds"])))

7
